In [ ]:
# ==============================================================================
# 1. SETUP E IMPORTAÇÕES
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# ==============================================================================
# 2. PREPARAÇÃO DO DATASET (VERSÃO TURBO 🚀)
# ==============================================================================
print("Carregando e processando dados...")

# Carregar dados
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')

# Merge simples
df = df_items.merge(df_products, on='product_id', how='left')
features = ['price', 'freight_value', 'product_weight_g', 'product_category_name']
df = df[features].dropna()

# --- MUDANÇA 1: REDUZIR DRASTICAMENTE A AMOSTRA ---
# De 20.000 para 5.000. Para fins acadêmicos, o resultado é o mesmo e roda em segundos.
df_positive = df.sample(n=5000, random_state=42).copy()
df_positive['target'] = 1

# Gerando exemplos negativos
df_negative = df_positive.copy()
df_negative['price'] = df_negative['price'].sample(frac=1).values
df_negative['product_category_name'] = df_negative['product_category_name'].sample(frac=1).values
df_negative['target'] = 0

# Juntando
df_final = pd.concat([df_positive, df_negative], axis=0)

# Encoding (Rápido)
le_cat = LabelEncoder()
df_final['category_encoded'] = le_cat.fit_transform(df_final['product_category_name'])

X = df_final[['price', 'freight_value', 'product_weight_g', 'category_encoded']]
y = df_final['target']

# Divisão
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Normalização
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados prontos! Iniciando treinamento...")

# ==============================================================================
# 3. TREINAMENTO OTIMIZADO (COM n_jobs=-1)
# ==============================================================================

# --- MODELO A: KNN ---
# n_jobs=-1 usa todos os núcleos do processador
print("\n1. Treinando KNN...")
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1) 
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# --- MODELO B: Random Forest (Versão Leve) ---
# n_estimators=50 (menos árvores = mais rápido)
# n_jobs=-1 (processamento paralelo)
print("2. Treinando Random Forest...")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# ==============================================================================
# 4. RESULTADOS RÁPIDOS
# ==============================================================================

def mostrar_resultado(nome, y_real, y_pred):
    acc = accuracy_score(y_real, y_pred)
    prec = precision_score(y_real, y_pred)
    rec = recall_score(y_real, y_pred)
    print(f"\n>> {nome}: Acurácia: {acc:.2f} | Precision: {prec:.2f} | Recall: {rec:.2f}")

mostrar_resultado("KNN", y_test, y_pred_knn)
mostrar_resultado("Random Forest", y_test, y_pred_rf)

# Gráfico simples
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matriz Confusão (RF)')
plt.show()

print("\nConcluído com sucesso!")